# Clustering models for neighborhood segmentation

This notebook determines the optimal number of clusters using the elbow
method and silhouette scores, fits KMeans and Agglomerative clustering
models, and compares results to select the best segmentation.

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import silhouette_score, silhouette_samples

sys.path.insert(0, '..')
from src.data_loader import build_feature_matrix, FEATURE_COLUMNS
from src.model import (
    compute_elbow, find_optimal_k, train_kmeans,
    train_agglomerative, fit_pca, save_model
)

pd.set_option('display.max_columns', 50)

## 1. Prepare the scaled feature matrix

In [ ]:
raw_df, scaled_df, scaler = build_feature_matrix()
X = scaled_df[FEATURE_COLUMNS].values

print(f"Feature matrix shape: {X.shape}")
print(f"Communities: {len(raw_df)}")

## 2. Elbow method and silhouette analysis

The elbow method plots inertia (within-cluster sum of squares) against
the number of clusters. The silhouette score measures how well each
point fits its assigned cluster relative to other clusters; higher is
better.

In [ ]:
elbow_df = compute_elbow(X, k_range=range(2, 11))
print(elbow_df.to_string(index=False))

In [ ]:
fig = make_subplots(specs=[[{'secondary_y': True}]])

fig.add_trace(
    go.Scatter(x=elbow_df['k'], y=elbow_df['inertia'],
               mode='lines+markers', name='Inertia'),
    secondary_y=False
)
fig.add_trace(
    go.Scatter(x=elbow_df['k'], y=elbow_df['silhouette'],
               mode='lines+markers', name='Silhouette', line=dict(color='green')),
    secondary_y=True
)

fig.update_xaxes(title_text='Number of clusters (k)')
fig.update_yaxes(title_text='Inertia', secondary_y=False)
fig.update_yaxes(title_text='Silhouette score', secondary_y=True)
fig.update_layout(title='Elbow method with silhouette scores', height=400)
fig.show()

In [ ]:
optimal_k = find_optimal_k(elbow_df)
print(f"Optimal k by silhouette score: {optimal_k}")

## 3. KMeans clustering with multiple k values

We fit KMeans for k=3, k=4, and k=5 to compare how the segmentation
looks at different granularities.

In [ ]:
kmeans_models = {}
for k in [3, 4, 5]:
    km = train_kmeans(X, n_clusters=k)
    sil = silhouette_score(X, km.labels_)
    kmeans_models[k] = km
    print(f"k={k}: inertia={km.inertia_:.1f}, silhouette={sil:.4f}")

In [ ]:
# Visualise clusters in PCA space for each k
pca, X_pca = fit_pca(X, n_components=2)

fig = make_subplots(rows=1, cols=3,
                    subplot_titles=['k=3', 'k=4', 'k=5'])

for i, k in enumerate([3, 4, 5]):
    labels = kmeans_models[k].labels_
    for cl in sorted(set(labels)):
        mask = labels == cl
        fig.add_trace(
            go.Scatter(x=X_pca[mask, 0], y=X_pca[mask, 1],
                       mode='markers', name=f'Cluster {cl}',
                       marker=dict(size=5, opacity=0.6),
                       showlegend=(i == 0)),
            row=1, col=i + 1
        )

fig.update_layout(height=400, title_text='KMeans clusters in PCA space')
fig.show()

## 4. Agglomerative clustering

Agglomerative (hierarchical) clustering with Ward linkage provides an
alternative grouping. We compare its silhouette score to KMeans.

In [ ]:
agg_models = {}
for k in [3, 4, 5]:
    agg = train_agglomerative(X, n_clusters=k, linkage='ward')
    sil = silhouette_score(X, agg.labels_)
    agg_models[k] = agg
    print(f"Agglomerative k={k}: silhouette={sil:.4f}")

## 5. Compare silhouette scores

A side-by-side comparison of KMeans and Agglomerative clustering at
each k value helps choose both the algorithm and the granularity.

In [ ]:
comparison_rows = []
for k in [3, 4, 5]:
    km_sil = silhouette_score(X, kmeans_models[k].labels_)
    agg_sil = silhouette_score(X, agg_models[k].labels_)
    comparison_rows.append({'k': k, 'KMeans': km_sil, 'Agglomerative': agg_sil})

comp_df = pd.DataFrame(comparison_rows)
print(comp_df.to_string(index=False))

fig = go.Figure()
fig.add_trace(go.Bar(name='KMeans', x=comp_df['k'], y=comp_df['KMeans']))
fig.add_trace(go.Bar(name='Agglomerative', x=comp_df['k'], y=comp_df['Agglomerative']))
fig.update_layout(barmode='group',
                  title='Silhouette score comparison',
                  xaxis_title='k', yaxis_title='Silhouette score',
                  height=400)
fig.show()

## 6. Select optimal k and save models

In [ ]:
best_km = kmeans_models[optimal_k]
best_agg = agg_models[optimal_k]

print(f"Selected k={optimal_k}")
print(f"  KMeans silhouette:       {silhouette_score(X, best_km.labels_):.4f}")
print(f"  Agglomerative silhouette: {silhouette_score(X, best_agg.labels_):.4f}")

save_model(best_km, 'kmeans_model.joblib')
save_model(best_agg, 'agglomerative_model.joblib')
save_model(scaler, 'feature_scaler.joblib')
print("Models saved.")

In [ ]:
# Silhouette plot for the selected KMeans model
sil_vals = silhouette_samples(X, best_km.labels_)
sil_df = pd.DataFrame({'cluster': best_km.labels_, 'silhouette': sil_vals})

fig = px.histogram(
    sil_df, x='silhouette', color='cluster', nbins=40,
    title=f'Silhouette value distribution (k={optimal_k})',
    labels={'silhouette': 'Silhouette coefficient', 'cluster': 'Cluster'},
    barmode='overlay', opacity=0.7
)
fig.add_vline(x=silhouette_score(X, best_km.labels_), line_dash='dash',
              line_color='red', annotation_text='Mean silhouette')
fig.update_layout(height=400)
fig.show()

## 7. Summary

- The elbow method and silhouette analysis both point toward an optimal
  cluster count of k that balances compactness with separation.
- KMeans and Agglomerative clustering produce similar silhouette scores,
  but KMeans is preferred for its computational efficiency and interpretable
  centroids.
- The silhouette distribution confirms that most communities are well
  assigned, with only a few borderline cases.

The next notebook profiles each cluster to understand what makes the
segments distinct and provides geographic interpretation.